# 1. Conceptual Foundations

### 1.1 The Problem of Raw Text

Machines cannot consume text directly. A model needs numbers. The simplest way to convert text into numbers is to create a **document-term matrix (DTM)**:

* Each **row** is a document.
* Each **column** is a term (a unique word or token).
* Each **cell** contains some measure of how strongly that term is associated with that document.

The naïve option is **term frequency (TF)**: just counting how often each word appears in a document.

Example:
For two documents:

* *Doc1*: “hiking trail forest forest”
* *Doc2*: “mountain trail view”

The DTM with raw counts might look like:

| Document | forest | hiking | mountain | trail | view |
| -------- | ------ | ------ | -------- | ----- | ---- |
| Doc1     | 2      | 1      | 0        | 1     | 0    |
| Doc2     | 0      | 0      | 1        | 1     | 1    |

Problem: Common words dominate (think “the”, “and”, “is”), even though they are poor discriminators between documents.

### 1.2 The IDF Correction

To fix this, we introduce **inverse document frequency (IDF)**. IDF penalises terms that appear in many documents and rewards terms that are rare across the corpus.

The formula:

$$
\text{tf-idf}(t, d) = \text{tf}(t, d) \times \log \left(\frac{N}{1 + \text{df}(t)}\right)
$$

* $\text{tf}(t, d)$: frequency of term $t$ in document $d$.
* $\text{df}(t)$: number of documents containing term $t$.
* $N$: total number of documents.
* The log scaling ensures the penalty is not too harsh.
* The $+1$ in the denominator avoids division by zero.

Intuition:

* Words that appear **often in one document but rarely elsewhere** get large weights.
* Words that appear **in every document** get low weights, regardless of local frequency.

### 1.3 Why Feature Selection Is Needed

A corpus can contain tens of thousands of distinct terms. TF-IDF will happily create a column for each, yielding an enormous and very sparse matrix. Problems arise:

1. **Noise**: Many words are irrelevant (typos, rare terms, stopwords not removed).
2. **Efficiency**: High dimensionality makes models slow and costly.
3. **Overfitting**: Too many features allow models to fit spurious patterns.
4. **Interpretability**: We may want to see the top words that truly drive predictions.

Hence, selecting features is critical. We might want the **top-k features overall**, or the **top terms per document**, or we might filter by a **minimum threshold weight**.

### 1.4 Why This Matters in Machine Learning

* **Classification**: A logistic regression predicting spam will work better if “free” or “win” is preserved, but “the” is discarded.
* **Clustering**: Clusters form more clearly if irrelevant noise is dropped.
* **Interpretability**: Showing top terms to users or domain experts requires exactly the procedure your lecturer tried to demonstrate (extracting weights per document).



# Selecting informative features from TF-IDF vectors

## 2. The anatomy of TF-IDF in scikit-learn

When you run `TfidfVectorizer.fit_transform(corpus)`, scikit-learn returns a **CSR sparse matrix** `X` with shape `(n_documents, n_terms)`. Three arrays encode the non-zero values:

* `X.data`: the non-zero TF-IDF weights.
* `X.indices`: the column indices of those weights.
* `X.indptr`: row boundaries. For document `i`, the slice `X.indptr[i]:X.indptr[i+1]` tells you where its non-zeros live in `data` and `indices`.

There are two ways to map column indices back to words:

1. Modern and straightforward: `vectorizer.get_feature_names_out()` returns an array `feature_names`, where `feature_names[j]` is the term at column `j`.
2. Older pattern (what your lecturer hints at): reverse `vectorizer.vocabulary_` from `{term: index}` to `{index: term}`. This works, but the first method is cleaner.


In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load a small, real text dataset and build a TF-IDF matrix
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

categories = ["sci.med", "rec.sport.baseball"]
newsgroups = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "quotes")
)

In [3]:
tdidf_vec = TfidfVectorizer()

text_tfidf = tdidf_vec.fit_transform(newsgroups)

In [4]:
print(tdidf_vec.vocabulary_)

{'data': 0, 'filenames': 2, 'target_names': 4, 'target': 3, 'descr': 1}


In [5]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),  # unigrams + bigrams to capture short phrases
    min_df=5,  # coarse feature filtering by document frequency
    max_df=0.6,  # drop very common terms across docs
    dtype=np.float64,
)

X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target
feature_names = vectorizer.get_feature_names_out()

X.shape, len(feature_names), set(newsgroups.target_names)

((1191, 4375), 4375, {'rec.sport.baseball', 'sci.med'})

In [6]:
vectorizer.vocabulary_

{'looking': 2371,
 'source': 3684,
 'american': 419,
 'league': 2273,
 'baseball': 582,
 'stats': 3758,
 'individual': 2029,
 'players': 2991,
 'newspapers': 2705,
 'want': 4218,
 'provide': 3133,
 'list': 2335,
 'nice': 2709,
 'reports': 3317,
 '35': 149,
 'week': 4246,
 'does': 1297,
 'know': 2223,
 'statistics': 3757,
 'idea': 1979,
 'cost': 1059,
 'canada': 761,
 'systems': 3882,
 'toronto': 4016,
 'ontario': 2805,
 '416': 169,
 'american league': 420,
 'league baseball': 2275,
 'does know': 1299,
 'particularly': 2873,
 'world': 4311,
 'series': 3550,
 'season': 3518,
 'probably': 3089,
 'valuable': 4159,
 'say': 3481,
 'putting': 3163,
 'olerud': 2797,
 'pitch': 2963,
 'yeah': 4342,
 'getting': 1737,
 'sucked': 3827,
 'water': 4235,
 'jays': 2123,
 'won': 4296,
 'spite': 3711,
 'morris': 2619,
 'roger': 3395,
 'return': 3350,
 'days': 1144,
 'postings': 3031,
 'ago': 377,
 'poster': 3028,
 'valentine': 4157,
 'used': 4140,
 'ws': 4334,
 'rings': 3376,
 'measure': 2502,
 'better':

In [7]:
vocab = {v: k for k, v in vectorizer.vocabulary_.items()}

In [8]:
display(sorted(vocab.values()))

['00',
 '00 p003228',
 '000',
 '000 000',
 '01',
 '02',
 '03',
 '0358',
 '0358 athens',
 '04',
 '05',
 '06',
 '07',
 '08',
 '10',
 '10 12',
 '10 15',
 '10 usenet',
 '10 years',
 '100',
 '1000',
 '101',
 '104',
 '108',
 '109',
 '11',
 '110',
 '114',
 '11th',
 '12',
 '120',
 '125',
 '128',
 '129',
 '13',
 '134',
 '135',
 '14',
 '141',
 '143',
 '144',
 '147',
 '149',
 '15',
 '15 day',
 '15 years',
 '150',
 '156',
 '16',
 '161',
 '162',
 '167',
 '17',
 '172',
 '17505',
 '17505 nw',
 '18',
 '19',
 '190',
 '1964',
 '1968',
 '1968 tanstaafl',
 '1976',
 '1979',
 '1980',
 '1981',
 '1982',
 '1983',
 '1984',
 '1985',
 '1986',
 '1987',
 '1988',
 '1989',
 '1990',
 '1991',
 '1992',
 '1993',
 '1993 rap',
 '19th',
 '1b',
 '1st',
 '20',
 '20 minutes',
 '20 years',
 '200',
 '2000',
 '202',
 '206',
 '20th',
 '21',
 '215',
 '219',
 '22',
 '222',
 '225',
 '23',
 '236',
 '238',
 '24',
 '24 hours',
 '240',
 '2400x4',
 '243',
 '245',
 '245 3205',
 '245 4366',
 '25',
 '250',
 '26',
 '262',
 '267',
 '268',
 '27

In [9]:
zipped_row = sorted(dict(zip(X[3].indices, X[3].data)))
display(zipped_row)

[np.int32(19),
 np.int32(571),
 np.int32(572),
 np.int32(742),
 np.int32(743),
 np.int32(818),
 np.int32(837),
 np.int32(838),
 np.int32(1096),
 np.int32(1360),
 np.int32(1361),
 np.int32(1397),
 np.int32(1409),
 np.int32(1555),
 np.int32(1714),
 np.int32(1715),
 np.int32(1775),
 np.int32(1776),
 np.int32(2072),
 np.int32(2073),
 np.int32(2238),
 np.int32(2399),
 np.int32(2651),
 np.int32(2652),
 np.int32(2726),
 np.int32(2882),
 np.int32(2970),
 np.int32(2971),
 np.int32(3568),
 np.int32(3569),
 np.int32(3628),
 np.int32(3629),
 np.int32(3672),
 np.int32(3850),
 np.int32(3865),
 np.int32(3866),
 np.int32(4189)]

## 3. Inspecting per-document word weights

We often want “the most salient terms in this specific document”. Use the CSR pointers to avoid dense conversions.


In [10]:
# Inspect top TF-IDF terms for a single document using CSR internals

doc_id = 7  # pick any document index
row_start, row_end = X.indptr[doc_id], X.indptr[doc_id + 1]
row_indices = X.indices[row_start:row_end]
row_weights = X.data[row_start:row_end]

# Sort terms by weight (descending)
order = np.argsort(row_weights)[::-1]
top_idx = row_indices[order]
top_wts = row_weights[order]

# Build (term, weight) pairs for the top 15
top_terms = [(feature_names[j], float(w)) for j, w in zip(top_idx[:15], top_wts[:15])]
top_terms

[('jewish', 0.4390996723379741),
 ('stone', 0.2586888300109145),
 ('edu sorry', 0.1431339468524937),
 ('look like', 0.1431339468524937),
 ('hof', 0.1431339468524937),
 ('carew', 0.1431339468524937),
 ('stuck', 0.1431339468524937),
 ('leagues', 0.1339096542299725),
 ('harry', 0.1339096542299725),
 ('converted', 0.1339096542299725),
 ('thf2 kimbark', 0.13151271252561195),
 ('kimbark uchicago', 0.13151271252561195),
 ('thf2', 0.13151271252561195),
 ('kimbark', 0.13151271252561195),
 ('law school', 0.13151271252561195)]

## 4. Selecting features within a document

You have two natural levers: top-k by rank, or threshold by weight. Both are legitimate, but they answer different questions.

In [11]:
# Top-k terms for a document
k = 20
top_k_terms = [feature_names[j] for j in top_idx[:k]]

# Threshold by weight (keep terms with TF-IDF >= 0.25, for example)
threshold = 0.25
kept_terms = [
    feature_names[j] for j, w in zip(row_indices, row_weights) if w >= threshold
]


* **Top-k** gives you a fixed budget of words per document.
* **Threshold** is scale-dependent. It keeps terms that are genuinely strong in that document, but the count may vary widely.

Use this only if you must mirror the lecture. Prefer `get_feature_names_out()` in new code.

In [12]:
# Optional: build a reversed vocabulary mapping {col_index: term}
rev_vocab = {idx: term for term, idx in vectorizer.vocabulary_.items()}
{rev_vocab[j]: float(w) for j, w in zip(row_indices, row_weights)}

{'just': 0.0578330859545169,
 'al': 0.09729509488763968,
 'like': 0.057043992811216464,
 'pitcher': 0.1017482653888141,
 'long': 0.07823685329863374,
 'edu': 0.04942624312234345,
 'jewish': 0.4390996723379741,
 'hof': 0.1431339468524937,
 'er': 0.12934441500545724,
 'carew': 0.1431339468524937,
 'converted': 0.1339096542299725,
 'major': 0.08692074272507802,
 'leagues': 0.1339096542299725,
 'cy': 0.1194409703123008,
 'young': 0.09314729682159151,
 'steve': 0.08567386056075794,
 'stone': 0.2586888300109145,
 'koufax': 0.12554395047742342,
 'ken': 0.12082017468915508,
 'wrong': 0.09489815318327914,
 'thinking': 0.10237158944411272,
 'threw': 0.1194409703123008,
 'hitter': 0.09886467614811614,
 'nl': 0.10301247421416432,
 'big': 0.08692074272507802,
 'ed': 0.11463370854104604,
 'quite': 0.08964038352559496,
 'starting': 0.10577342216145028,
 'rotation': 0.10977491808449352,
 'catch': 0.11814062183704262,
 'harry': 0.1339096542299725,
 '3b': 0.12554395047742342,
 'chance': 0.10505114229622


## 5. Selecting features across the corpus

Document-level choices help with **interpretation**. For modelling you usually want **corpus-level feature selection** that is performed **within cross-validation** to avoid leakage. Three principled families are common and work well with TF-IDF:

### 5.1 Univariate statistical tests (fast, strong baselines)

`chi2` and `mutual_info_classif` evaluate the association between each feature and the class labels. They work on the sparse matrix, and they are scale-agnostic in a way that suits TF-IDF.

In [13]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

pipe_chi2 = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english", min_df=5, max_df=0.6, ngram_range=(1, 2)
            ),
        ),
        ("kbest", SelectKBest(score_func=chi2, k=2000)),  # tune k
        (
            "clf",
            LogisticRegression(
                max_iter=2000, n_jobs=1, solver="liblinear", random_state=42
            ),
        ),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe_chi2, newsgroups.data, y, cv=cv, scoring="f1_macro", n_jobs=-1
)
scores.mean(), scores.std()

(np.float64(0.9731352335565047), np.float64(0.007755630990475559))


Key points:

* Feature selection sits **inside** the pipeline. The split happens first. Each fold computes its own TF-IDF and its own best features. No leakage.
* `k` is a hyperparameter. Search it, do not guess it.

## 5.2 Embedded selection via sparse regularisation (L1)

L1-penalised linear models shrink many coefficients exactly to zero, yielding a **selected** feature set. This often performs competitively and gives you weights for interpretation.

In [14]:
from sklearn.feature_selection import SelectFromModel

pipe_l1 = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english", min_df=5, max_df=0.6, ngram_range=(1, 2)
            ),
        ),
        (
            "select",
            SelectFromModel(
                LogisticRegression(
                    penalty="l1", solver="liblinear", max_iter=2000, random_state=42
                )
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=2000, n_jobs=1, solver="liblinear", random_state=42
            ),
        ),
    ]
)

scores = cross_val_score(
    pipe_l1, newsgroups.data, y, cv=cv, scoring="f1_macro", n_jobs=1
)
scores.mean(), scores.std()

(np.float64(0.8968868069064208), np.float64(0.027294569855457095))

* You can tune `C` on the inner `LogisticRegression` inside `SelectFromModel` to control sparsity. Smaller `C` means stronger regularisation and fewer features.

### 5.3 Hybrid: univariate filter + sparse model

Use `SelectKBest` to prune the feature space to a few thousand columns, then fit an L1 model. This combination is strong and efficient.



In [15]:
from sklearn.model_selection import GridSearchCV

pipe_hybrid = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english", min_df=5, max_df=0.6, ngram_range=(1, 2)
            ),
        ),
        ("kbest", SelectKBest(score_func=chi2)),
        (
            "clf",
            LogisticRegression(
                penalty="l1", solver="liblinear", max_iter=2000, random_state=42
            ),
        ),
    ]
)

param_grid = {"kbest__k": [1000, 2000, 3000], "clf__C": [0.2, 0.5, 1.0, 2.0]}

search = GridSearchCV(
    pipe_hybrid, param_grid=param_grid, scoring="f1_macro", cv=cv, n_jobs=-1
)
search.fit(newsgroups.data, y)
search.best_params_, search.best_score_

({'clf__C': 2.0, 'kbest__k': 1000}, np.float64(0.9174106868240821))

This is the principled alternative to “take the top 20 percent of TF-IDF weights blindly”.


In [16]:
# Fit once (e.g., best params from search) and inspect selected terms
best_pipe = search.best_estimator_
best_pipe.fit(newsgroups.data, y)

tfidf = best_pipe.named_steps["tfidf"]
feature_names = tfidf.get_feature_names_out()

kbest = best_pipe.named_steps["kbest"]
mask = kbest.get_support()  # boolean mask of kept columns
kept_terms = feature_names[mask]  # strings of selected features

len(kept_terms), kept_terms[:20]

(1000,
 array(['01', '02', '03', '15 day', '162', '1964', '1993', '1993 rap',
        '2b', '2nd', '300', '333', '35', '3b', '3rd', '411', '500', '534',
        '604 245', '619'], dtype=object))

For the L1 approach with `SelectFromModel`:

In [17]:
pipe_l1.fit(newsgroups.data, y)
tfidf = pipe_l1.named_steps["tfidf"]
feature_names = tfidf.get_feature_names_out()

selector = pipe_l1.named_steps["select"]
mask = selector.get_support()
l1_terms = feature_names[mask]
len(l1_terms), l1_terms[:20]

(67,
 array(['1993', 'alomar', 'ball', 'baseball', 'blood', 'bob', 'body',
        'braves', 'cancer', 'case', 'cause', 'colorado', 'cubs', 'dave',
        'did', 'disease', 'dl', 'doctor', 'duke', 'dyer'], dtype=object))

You can also recover the **coefficient magnitudes** from the classifier for ranking:


In [18]:
clf = best_pipe.named_steps["clf"]
coefs = np.abs(clf.coef_)  # shape: (n_classes, n_features_kept)
global_importance = coefs.mean(axis=0)
ranked_terms = sorted(
    zip(kept_terms, global_importance), key=lambda t: t[1], reverse=True
)[:30]
ranked_terms[:10]

[('gordon banks', np.float64(12.9386380027108)),
 ('game', np.float64(11.920343812758926)),
 ('team', np.float64(11.896604488101625)),
 ('baseball', np.float64(11.259951310471319)),
 ('games', np.float64(8.10667819829345)),
 ('year', np.float64(7.773501104888734)),
 ('health', np.float64(7.688743020966713)),
 ('disease', np.float64(7.568652605698526)),
 ('doctor', np.float64(7.099843940373643)),
 ('season', np.float64(6.629420214547323))]

This gives a defensible “top features overall” list grounded in the trained model.


## 6. Extracting selected features and inspecting them

Interpreting a fitted pipeline is crucial. After fitting, recover the **indices of selected features** in the final vectoriser space, then map back to strings.


## 7. Extensions, trade-offs, and pitfalls

* **Vectoriser hyperparameters are feature selection**: `min_df`, `max_df`, `ngram_range`, and `max_features` profoundly change the space. Tuning them belongs in the grid.
* **Normalisation matters**: `TfidfVectorizer` normalises rows by default (`norm="l2"`). This affects absolute TF-IDF numbers and thus thresholding strategies.
* **Leakage is the classic error**: Never fit TF-IDF or select features on the full dataset before CV. Always wrap in a `Pipeline`.
* **Class imbalance**: Choose scoring metrics that reflect your problem (macro F1 for balanced attention across classes).
* **When TF-IDF is not ideal**: For semantic tasks, embeddings or transformer encoders may outperform sparse text vectors. For linear, high-dimensional problems with limited compute, TF-IDF + linear model is still exceptionally strong and transparent.

# Putting it all together on a tiny end-to-end run

The following is compact and intentionally “beginner-kind” but rigorous. It loads data, builds a robust pipeline, performs a small search over both vectoriser and selector, and reports the best configuration.

In [19]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV

categories = ["sci.med", "rec.sport.baseball"]
data = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "quotes")
)
X_text, y = data.data, data.target

pipe = Pipeline(
    [
        ("tfidf", TfidfVectorizer(stop_words="english")),
        ("kbest", SelectKBest(score_func=chi2)),
        (
            "clf",
            LogisticRegression(
                penalty="l2", solver="liblinear", max_iter=2000, random_state=42
            ),
        ),
    ]
)

param_grid = {
    "tfidf__min_df": [3, 5],
    "tfidf__max_df": [0.5, 0.7],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "kbest__k": [1000, 2000, 2500],
    "clf__C": [0.5, 1.0, 2.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
search.fit(X_text, y)

search.best_params_, round(search.best_score_, 4)

({'clf__C': 2.0,
  'kbest__k': 2500,
  'tfidf__max_df': 0.5,
  'tfidf__min_df': 3,
  'tfidf__ngram_range': (1, 2)},
 np.float64(0.979))


Once you have the best pipeline, inspect the selected features and their model weights as shown above. That closes the loop: you have a **principled selection mechanism**, integrated with cross-validation, and you retain interpretability.